In [ ]:
# ==============================================================================
# 1. KURULUM VE ORTAM
# ==============================================================================
# Bu komut dosyası, en iyi performansı gösteren model olan DistilBERT + Features’ı eğitir ve değerlendirir;

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import time
import os
import psutil
from bs4 import BeautifulSoup
from tqdm.auto import tqdm
import pickle

In [ ]:
import tensorflow as tf
from transformers import DistilBertTokenizerFast, TFDistilBertModel
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
import string
from imblearn.over_sampling import RandomOverSampler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score
)

In [ ]:
# --- Ayarlamalar ---
RANDOM_STATE = 42
DATA_PATH = "../data/balanced_dataset.csv"
PRETRAINED_DISTILBERT_PATH = "distilbert-base-uncased"
MODEL_SAVE_DIR = "../saved_models/"
os.makedirs(MODEL_SAVE_DIR, exist_ok=True)

print(f"TensorFlow Version: {tf.__version__}")
from transformers import __version__ as transformers_version
print(f"Transformers Version: {transformers_version}")

In [ ]:
# ==============================================================================
# 2. VERİ YÜKLEME VE ÖN İŞLEME
# ==============================================================================
print("\n--- Veri Yükleme ve Ön İşleme Başlatıldı ---")

# Veri setini yükle
df = pd.read_csv(DATA_PATH)

# Gerekli sütunların varlığını kontrol et
required_columns = {"text", "label"}
missing_columns = required_columns.difference(df.columns)

if missing_columns:
    raise ValueError(f"Eksik sütunlar bulundu: {missing_columns}")

# Yalnızca gerekli sütunları kullan
df = df[["text", "label"]].copy()

# Metni veya etiketi eksik olan kayıtları kaldır
df.dropna(subset=["text", "label"], inplace=True)

# Metin sütununu string biçimine dönüştür
df["text"] = df["text"].astype(str)

# Etiketleri sayısal biçime dönüştür
df["label"] = pd.to_numeric(df["label"], errors="coerce")

# Sayısala dönüştürülemeyen etiketleri kaldır
df.dropna(subset=["label"], inplace=True)
df["label"] = df["label"].astype(int)

# Etiketlerin yalnızca 0 ve 1 olduğunu kontrol et
invalid_labels = set(df["label"].unique()) - {0, 1}

if invalid_labels:
    raise ValueError(
        f"Geçersiz etiketler bulundu: {invalid_labels}. "
        "Etiketler yalnızca 0=ham ve 1=spam olmalıdır."
    )

print(f"Veri seti başarıyla yüklendi. Toplam kayıt: {len(df)}")

print("\nEtiket sayıları:")
print(df["label"].value_counts().sort_index())

print("\nEtiket oranları:")
print(df["label"].value_counts(normalize=True).sort_index().round(4))

# ==============================================================================
# TEKRAR EDEN E-POSTALARIN KONTROLU
# ==============================================================================
# Amaç:
# Aynı e-postanın eğitim ve test kümelerine birlikte düşmesini önlemek.

df["cleaned_text"] = df["text"].progress_apply(
    clean_text_for_transformer
)

df = df[
    df["cleaned_text"].str.len() > 0
].reset_index(drop=True)

df["duplicate_key"] = (
    df["cleaned_text"]
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

before_duplicate_removal = len(df)

df = df.drop_duplicates(
    subset=["duplicate_key"],
    keep="first"
).reset_index(drop=True)

df.drop(columns=["duplicate_key"], inplace=True)

print(
    f"Tekrar eden kayıt sayısı: "
    f"{before_duplicate_removal - len(df)}"
)

print(
    f"Duplicate temizliği sonrası kayıt sayısı: "
    f"{len(df)}"
)


# ==============================================================================
# TRANSFORMER İÇİN HAFİF METİN TEMİZLEME
# ==============================================================================
def clean_text_for_transformer(text):
    """
    E-posta metnini DistilBERT için hazırlar.

    Yapılan işlemler:
    - HTML etiketlerini kaldırır.
    - Satır sonlarını ve fazla boşlukları düzenler.
    - Metnin doğal dil yapısını korur.

    Yapılmayan işlemler:
    - Stopword kaldırma
    - Lemmatization
    - Noktalama işaretlerini silme
    - Küçük harfe dönüştürme

    Gerekçe:
    DistilBERT kendi tokenizer yapısını kullanır.
    Aşırı temizlik bağlamsal ve yapısal bilgiyi azaltabilir.
    """
    if not isinstance(text, str):
        return ""

    # HTML etiketlerini kaldır, metin bölümleri arasına boşluk ekle
    text = BeautifulSoup(text, "html.parser").get_text(separator=" ")

    # Satır sonlarını ve fazla boşlukları tek boşluğa indir
    text = re.sub(r"\s+", " ", text).strip()

    return text


# tqdm ile işlem ilerlemesini göster
tqdm.pandas(desc="Metinler temizleniyor")

print("\nDistilBERT için hafif metin temizleme uygulanıyor...")

df["cleaned_text"] = df["text"].progress_apply(
    clean_text_for_transformer
)

# Temizlik sonrasında boş kalan kayıtları kaldır
df = df[df["cleaned_text"].str.len() > 0].reset_index(drop=True)

print("Metin temizleme tamamlandı.")
print(f"Temizlik sonrası toplam kayıt: {len(df)}")

In [ ]:
# ==============================================================================
# 3. ÖZELLİK ÇIKARIMI VE VERİ BÖLME
# ==============================================================================
print("\n--- Özellik Çıkarımı Başlatıldı ---")


# ==============================================================================
# 3.1. VERİYİ %70 EĞİTİM - %10 VALIDATION - %20 TEST OLARAK BÖLME
# ==============================================================================
# Amaç:
# - Sınıf oranlarını korumak
# - Test verisini eğitim sürecinden tamamen ayırmak
# - Veri sızıntısını önlemek

all_indices = np.arange(len(df))
labels = df["label"].to_numpy()

# İlk aşamada %70 eğitim, %30 geçici küme oluşturulur
train_indices, temp_indices = train_test_split(
    all_indices,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=labels
)

# Geçici %30 küme:
# - %10 validation
# - %20 test
val_indices, test_indices = train_test_split(
    temp_indices,
    test_size=2 / 3,
    random_state=RANDOM_STATE,
    stratify=labels[temp_indices]
)

print(f"Eğitim örnek sayısı: {len(train_indices)}")
print(f"Validation örnek sayısı: {len(val_indices)}")
print(f"Test örnek sayısı: {len(test_indices)}")

print("\nEğitim sınıf dağılımı:")
print(pd.Series(labels[train_indices]).value_counts(normalize=True))

print("\nValidation sınıf dağılımı:")
print(pd.Series(labels[val_indices]).value_counts(normalize=True))

print("\nTest sınıf dağılımı:")
print(pd.Series(labels[test_indices]).value_counts(normalize=True))

In [ ]:
# ==============================================================================
# 3.2. YAPISAL ÖZELLİKLERİN ÇIKARILMASI
# ==============================================================================
# Kullanılan dört yapısal özellik:
# 1. Metin uzunluğu
# 2. Kelime sayısı
# 3. Büyük harf oranı
# 4. Noktalama işareti sayısı

print("\nHam e-posta metinlerinden yapısal özellikler çıkarılıyor...")


def calculate_uppercase_ratio(text):
    """
    Büyük harflerin alfabetik karakterlere oranını hesaplar.

    Örnek:
    'FREE Offer' metninde yalnızca harfler dikkate alınır.
    Boşluklar ve noktalama işaretleri paydaya dahil edilmez.
    """
    if not isinstance(text, str):
        return 0.0

    alphabetic_characters = [
        character for character in text
        if character.isalpha()
    ]

    if len(alphabetic_characters) == 0:
        return 0.0

    uppercase_count = sum(
        character.isupper()
        for character in alphabetic_characters
    )

    return uppercase_count / len(alphabetic_characters)


def count_punctuation(text):
    """
    Metindeki noktalama işaretlerinin toplam sayısını hesaplar.
    """
    if not isinstance(text, str):
        return 0

    return sum(
        character in string.punctuation
        for character in text
    )


# Yapısal özellikler temizlenmemiş ham metinden çıkarılır
# Böylece büyük harf ve noktalama bilgisi korunur
df["text_len"] = df["text"].apply(len)

df["word_count"] = df["text"].apply(
    lambda text: len(text.split())
)

df["uppercase_ratio"] = df["text"].apply(
    calculate_uppercase_ratio
)

df["punctuation_count"] = df["text"].apply(
    count_punctuation
)


numerical_feature_cols = [
    "text_len",
    "word_count",
    "uppercase_ratio",
    "punctuation_count"
]

numerical_features = df[
    numerical_feature_cols
].to_numpy(dtype=np.float32)

print("Yapısal özellikler çıkarıldı.")
print(f"Yapısal özellik matrisi: {numerical_features.shape}")

In [ ]:
# ==============================================================================
# 3.3. YAPISAL ÖZELLİKLERİN ÖLÇEKLENMESİ
# ==============================================================================
# Kritik nokta:
# Scaler yalnızca eğitim verisine fit edilir.
# Validation ve test verileri yalnızca transform edilir.

scaler = StandardScaler()

# Eğitim verisinden ortalama ve standart sapma öğrenilir
train_numerical_scaled = scaler.fit_transform(
    numerical_features[train_indices]
)

# Validation ve test verilerine aynı dönüşüm uygulanır
val_numerical_scaled = scaler.transform(
    numerical_features[val_indices]
)

test_numerical_scaled = scaler.transform(
    numerical_features[test_indices]
)

# Eğitilmiş scaler ileride kullanılmak üzere kaydedilir
scaler_path = os.path.join(
    MODEL_SAVE_DIR,
    "scaler.pkl"
)

with open(scaler_path, "wb") as file:
    pickle.dump(scaler, file)

print("Yapısal özellikler ölçeklendirildi.")
print("Scaler yalnızca eğitim verisine fit edildi.")
print(f"Scaler kaydedildi: {scaler_path}")

In [ ]:
# ==============================================================================
# 3.4. DISTILBERT MODELİ VE TOKENIZER
# ==============================================================================
# Bu bölümde DistilBERT'in ağırlıkları güncellenmez.
# Model yalnızca 768 boyutlu anlamsal embedding üretir.
# Bu nedenle işlem fine-tuning değil, özellik çıkarımıdır.

print("\nDistilBERT modeli ve tokenizer yükleniyor...")

tokenizer = DistilBertTokenizerFast.from_pretrained(
    PRETRAINED_DISTILBERT_PATH
)

distilbert_base = TFDistilBertModel.from_pretrained(
    PRETRAINED_DISTILBERT_PATH
)

MAX_LEN = 256
BATCH_SIZE_EMBEDDING = 32

print("DistilBERT ve tokenizer başarıyla yüklendi.")

In [ ]:
# ==============================================================================
# 3.5. DISTILBERT EMBEDDING ÜRETME FONKSİYONU
# ==============================================================================
def generate_distilbert_embeddings(
    texts,
    tokenizer,
    model,
    max_length=256,
    batch_size=32
):
    """
    Metinleri DistilBERT ile 768 boyutlu anlamsal vektörlere dönüştürür.

    İşlem sırası:
    - Tokenization
    - Padding
    - Truncation
    - DistilBERT çıkarımı
    - İlk token vektörünün alınması
    """

    embeddings = []
    texts = list(texts)

    for start_index in tqdm(
        range(0, len(texts), batch_size),
        desc="DistilBERT embedding üretiliyor"
    ):
        batch_texts = texts[
            start_index:start_index + batch_size
        ]

        encoded_inputs = tokenizer(
            batch_texts,
            return_tensors="tf",
            truncation=True,
            padding="max_length",
            max_length=max_length
        )

        # training=False:
        # DistilBERT ağırlıkları güncellenmez
        outputs = model(
            input_ids=encoded_inputs["input_ids"],
            attention_mask=encoded_inputs["attention_mask"],
            training=False
        )

        # Her e-posta için ilk token temsili alınır
        # Çıktı boyutu: 768
        batch_embeddings = (
            outputs.last_hidden_state[:, 0, :]
            .numpy()
        )

        embeddings.append(batch_embeddings)

    return np.vstack(embeddings).astype(np.float32)

In [ ]:
# ==============================================================================
# 3.6. VERİ KÜMELERİ İÇİN EMBEDDING ÜRETİMİ
# ==============================================================================
print("\nEğitim embeddingleri oluşturuluyor...")

train_text_embeddings = generate_distilbert_embeddings(
    texts=df.iloc[train_indices]["cleaned_text"],
    tokenizer=tokenizer,
    model=distilbert_base,
    max_length=MAX_LEN,
    batch_size=BATCH_SIZE_EMBEDDING
)

print("\nValidation embeddingleri oluşturuluyor...")

val_text_embeddings = generate_distilbert_embeddings(
    texts=df.iloc[val_indices]["cleaned_text"],
    tokenizer=tokenizer,
    model=distilbert_base,
    max_length=MAX_LEN,
    batch_size=BATCH_SIZE_EMBEDDING
)

print("\nTest embeddingleri oluşturuluyor...")

test_text_embeddings = generate_distilbert_embeddings(
    texts=df.iloc[test_indices]["cleaned_text"],
    tokenizer=tokenizer,
    model=distilbert_base,
    max_length=MAX_LEN,
    batch_size=BATCH_SIZE_EMBEDDING
)

print("\nEmbedding üretimi tamamlandı.")
print(f"Eğitim embedding boyutu: {train_text_embeddings.shape}")
print(f"Validation embedding boyutu: {val_text_embeddings.shape}")
print(f"Test embedding boyutu: {test_text_embeddings.shape}")

In [ ]:
# ==============================================================================
# 3.7. SEMANTİK VE YAPISAL ÖZELLİKLERİN BİRLEŞTİRİLMESİ
# ==============================================================================
# DistilBERT embedding: 768 özellik
# Yapısal özellikler: 4 özellik
# Toplam: 772 özellik

X_train = np.concatenate(
    [
        train_text_embeddings,
        train_numerical_scaled
    ],
    axis=1
)

X_val = np.concatenate(
    [
        val_text_embeddings,
        val_numerical_scaled
    ],
    axis=1
)

X_test = np.concatenate(
    [
        test_text_embeddings,
        test_numerical_scaled
    ],
    axis=1
)

y_train = labels[train_indices]
y_val = labels[val_indices]
y_test = labels[test_indices]

# Boyut kontrolü
assert X_train.shape[1] == 772
assert X_val.shape[1] == 772
assert X_test.shape[1] == 772

print("\nÖzellik birleştirme tamamlandı.")

print(f"Eğitim kümesi: {X_train.shape}")
print(f"Validation kümesi: {X_val.shape}")
print(f"Test kümesi: {X_test.shape}")

print("\nHer örnek için:")
print("- 768 DistilBERT özelliği")
print("- 4 yapısal özellik")
print("- Toplam 772 özellik")

In [ ]:
# ==============================================================================
# 3.8. RANDOM OVERSAMPLING
# ==============================================================================
# Amaç:
# Eğitim kümesindeki sınıf dengesizliğini gidermek.
#
# Kritik nokta:
# Oversampling yalnızca eğitim kümesine uygulanır.
# Validation ve test kümeleri doğal dağılımında bırakılır.

print("\n--- Random Oversampling Uygulanıyor ---")

print("\nOversampling öncesi eğitim dağılımı:")
print(pd.Series(y_train).value_counts())

ros = RandomOverSampler(
    random_state=RANDOM_STATE
)

X_train_balanced, y_train_balanced = ros.fit_resample(
    X_train,
    y_train
)

print("\nOversampling sonrası eğitim dağılımı:")
print(pd.Series(y_train_balanced).value_counts())

print("\nValidation dağılımı:")
print(pd.Series(y_val).value_counts())

print("\nTest dağılımı:")
print(pd.Series(y_test).value_counts())

In [ ]:
# ==============================================================================
# 4.1. MLP SINIFLANDIRICI MİMARİSİ
# ==============================================================================
# Amaç:
# - 772 boyutlu hibrit özellik vektörünü sınıflandırmak
# - E-postayı ham (0) veya spam (1) olarak tahmin etmek
#
# Girdi:
# - 768 DistilBERT özelliği
# - 4 yapısal özellik
# - Toplam 772 özellik

print("\n--- MLP Sınıflandırıcı Oluşturuluyor ---")


def build_classifier(input_dim):
    """
    Hibrit özellik vektörünü kullanan MLP sınıflandırıcısını oluşturur.

    Katmanlar:
    - 128 nöronlu gizli katman
    - 64 nöronlu gizli katman
    - 1 nöronlu sigmoid çıkış katmanı
    """

    model = tf.keras.Sequential([

        # Girdi katmanı
        tf.keras.layers.Input(
            shape=(input_dim,),
            name="hybrid_feature_input"
        ),

        # Birinci gizli katman
        tf.keras.layers.Dense(
            units=128,
            activation="relu",
            kernel_regularizer=tf.keras.regularizers.l2(0.001),
            name="dense_128"
        ),

        # Eğitimi daha kararlı hâle getirir
        tf.keras.layers.BatchNormalization(
            name="batch_normalization_1"
        ),

        # Overfitting riskini azaltır
        tf.keras.layers.Dropout(
            rate=0.4,
            name="dropout_1"
        ),

        # İkinci gizli katman
        tf.keras.layers.Dense(
            units=64,
            activation="relu",
            kernel_regularizer=tf.keras.regularizers.l2(0.001),
            name="dense_64"
        ),

        tf.keras.layers.BatchNormalization(
            name="batch_normalization_2"
        ),

        tf.keras.layers.Dropout(
            rate=0.4,
            name="dropout_2"
        ),

        # Spam olasılığını 0-1 arasında üretir
        tf.keras.layers.Dense(
            units=1,
            activation="sigmoid",
            name="spam_probability"
        )
    ])

    return model

In [ ]:
# ==============================================================================
# 4.2. MODELİN DERLENMESİ
# ==============================================================================
# Adam:
# - Adaptif öğrenme oranı kullanır
#
# Binary Cross-Entropy:
# - İkili sınıflandırma için uygundur

LEARNING_RATE = 5e-5

classifier = build_classifier(
    input_dim=X_train.shape[1]
)

optimizer = tf.keras.optimizers.Adam(
    learning_rate=LEARNING_RATE
)

classifier.compile(
    optimizer=optimizer,

    # Ham-spam ikili sınıflandırma kaybı
    loss="binary_crossentropy",

    # Eğitim sırasında izlenecek metrikler
    metrics=[
        tf.keras.metrics.BinaryAccuracy(
            name="accuracy"
        ),
        tf.keras.metrics.Precision(
            name="precision"
        ),
        tf.keras.metrics.Recall(
            name="recall"
        ),
        tf.keras.metrics.AUC(
            name="roc_auc"
        ),
        tf.keras.metrics.AUC(
            name="pr_auc",
            curve="PR"
        )
    ]
)

classifier.summary()

In [ ]:
# ==============================================================================
# 4.3. EĞİTİM CALLBACKLERİ
# ==============================================================================
# EarlyStopping:
# - Validation loss iyileşmezse eğitimi durdurur
#
# ReduceLROnPlateau:
# - Öğrenme yavaşladığında learning rate'i düşürür
#
# ModelCheckpoint:
# - En iyi modeli kaydeder

MAX_EPOCHS = 15
BATCH_SIZE = 32
PATIENCE = 3


early_stopping = tf.keras.callbacks.EarlyStopping(

    # Validation kaybı izlenir
    monitor="val_loss",

    # Daha düşük değer daha iyidir
    mode="min",

    # 3 epoch boyunca iyileşme olmazsa dur
    patience=PATIENCE,

    # Çok küçük değişimler iyileşme sayılmaz
    min_delta=1e-4,

    # Eğitim sonunda en iyi ağırlıkları geri getir
    restore_best_weights=True,

    verbose=1
)


reduce_learning_rate = tf.keras.callbacks.ReduceLROnPlateau(

    monitor="val_loss",
    mode="min",

    # Learning rate yarıya düşürülür
    factor=0.5,

    # 1 epoch iyileşme olmazsa düşür
    patience=1,

    # Alt sınır
    min_lr=1e-7,

    verbose=1
)


checkpoint_path = os.path.join(
    MODEL_SAVE_DIR,
    "best_distilbert_features_classifier.keras"
)


model_checkpoint = tf.keras.callbacks.ModelCheckpoint(

    filepath=checkpoint_path,

    # En iyi validation kaybına sahip modeli kaydet
    monitor="val_loss",
    mode="min",

    save_best_only=True,
    save_weights_only=False,

    verbose=1
)

In [ ]:
# ==============================================================================
# 4.4. EĞİTİM ÖNCESİ KONTROLLER
# ==============================================================================

assert X_train_balanced.shape[0] == y_train_balanced.shape[0], \
    "Dengelenmiş eğitim verisi ve etiket sayısı eşleşmiyor."

assert X_val.shape[0] == y_val.shape[0], \
    "Validation verisi ve etiket sayısı eşleşmiyor."

assert X_test.shape[0] == y_test.shape[0], \
    "Test verisi ve etiket sayısı eşleşmiyor."

assert X_train_balanced.shape[1] == 772, \
    "Eğitim özellik boyutu 772 olmalıdır."

assert X_val.shape[1] == 772, \
    "Validation özellik boyutu 772 olmalıdır."

assert X_test.shape[1] == 772, \
    "Test özellik boyutu 772 olmalıdır."

assert not np.isnan(X_train_balanced).any(), \
    "Dengelenmiş eğitim verisinde NaN bulundu."

assert not np.isnan(X_val).any(), \
    "Validation verisinde NaN bulundu."

assert not np.isnan(X_test).any(), \
    "Test verisinde NaN bulundu."

assert np.isfinite(X_train_balanced).all(), \
    "Dengelenmiş eğitim verisinde sonsuz değer bulundu."

assert np.isfinite(X_val).all(), \
    "Validation verisinde sonsuz değer bulundu."

assert np.isfinite(X_test).all(), \
    "Test verisinde sonsuz değer bulundu."

print("Eğitim öncesi kontroller tamamlandı.")
print(f"Dengelenmiş eğitim: {X_train_balanced.shape}")
print(f"Validation: {X_val.shape}")
print(f"Test: {X_test.shape}")

In [ ]:
# ==============================================================================
# 4.5. MODELİN EĞİTİLMESİ
# ==============================================================================
print("\n--- Model Eğitimi Başlatılıyor ---")

# Mevcut Python işleminin bellek kullanımını izler
process = psutil.Process(os.getpid())

memory_before_mb = (
    process.memory_info().rss / (1024 ** 2)
)

training_start_time = time.time()


history = classifier.fit(
    X_train_balanced,
    y_train_balanced,
    validation_data=(X_val, y_val),
    epochs=15,
    batch_size=32,
    callbacks=[
        early_stopping,
        reduce_learning_rate,
        model_checkpoint
    ],
    verbose=1,
    shuffle=True
)


training_end_time = time.time()

memory_after_mb = (
    process.memory_info().rss / (1024 ** 2)
)

training_time_seconds = (
    training_end_time - training_start_time
)

print("\nModel eğitimi tamamlandı.")
print(
    f"Eğitim süresi: "
    f"{training_time_seconds:.2f} saniye"
)
print(
    f"Eğitim süresi: "
    f"{training_time_seconds / 60:.2f} dakika"
)

In [ ]:
# ==============================================================================
# 4.6. CPU BELLEK KULLANIMININ RAPORLANMASI
# ==============================================================================
memory_change_mb = (
    memory_after_mb - memory_before_mb
)

maximum_observed_memory_mb = max(
    memory_before_mb,
    memory_after_mb
)

print(
    f"Eğitim öncesi CPU belleği: "
    f"{memory_before_mb:.2f} MB"
)

print(
    f"Eğitim sonrası CPU belleği: "
    f"{memory_after_mb:.2f} MB"
)

print(
    f"Gözlenen bellek değişimi: "
    f"{memory_change_mb:.2f} MB"
)

print(
    f"Başlangıç ve bitişte gözlenen "
    f"en yüksek CPU belleği: "
    f"{maximum_observed_memory_mb:.2f} MB"
)

In [ ]:
# ==============================================================================
# 4.7. EĞİTİLMİŞ MODELİN KAYDEDİLMESİ
# ==============================================================================
# .keras formatı yeni TensorFlow sürümlerinde önerilen formattır

final_model_path = os.path.join(
    MODEL_SAVE_DIR,
    "distilbert_features_classifier.keras"
)

classifier.save(final_model_path)

print(f"Model kaydedildi: {final_model_path}")
print(f"En iyi model kaydedildi: {checkpoint_path}")

In [ ]:
# ==============================================================================
# 5.1. KARAR EŞİĞİNİN VALIDATION KÜMESİNDE BELİRLENMESİ
# ==============================================================================
# Amaç:
# - Spam sınıflandırması için en uygun olasılık eşiğini belirlemek
#
# Kritik nokta:
# - Eşik test kümesinde seçilmez
# - Test kümesi yalnızca nihai değerlendirme için kullanılır

print("\n--- Karar Eşiği Validation Kümesinde Belirleniyor ---")

# Validation kümesi için spam olasılıklarını üret
y_val_prob = classifier.predict(
    X_val,
    verbose=0
).flatten()

# Denenecek eşikler
threshold_values = np.arange(
    0.05,
    0.96,
    0.01
)

threshold_results = []

for threshold in threshold_values:

    # Olasılıkları sınıf etiketine dönüştür
    y_val_pred = (
        y_val_prob >= threshold
    ).astype(int)

    threshold_results.append({
        "threshold": threshold,
        "accuracy": accuracy_score(
            y_val,
            y_val_pred
        ),
        "precision": precision_score(
            y_val,
            y_val_pred,
            zero_division=0
        ),
        "recall": recall_score(
            y_val,
            y_val_pred,
            zero_division=0
        ),
        "f1_score": f1_score(
            y_val,
            y_val_pred,
            zero_division=0
        )
    })

threshold_results_df = pd.DataFrame(
    threshold_results
)

# En yüksek F1-Skoruna sahip eşik seçilir
best_threshold_row = threshold_results_df.loc[
    threshold_results_df["f1_score"].idxmax()
]

best_threshold = float(
    best_threshold_row["threshold"]
)

print(
    f"Seçilen karar eşiği: "
    f"{best_threshold:.2f}"
)

print("\nValidation sonuçları:")
print(best_threshold_row)

In [ ]:
# ==============================================================================
# 5.2. TEST KÜMESİ İÇİN TAHMİN ÜRETİMİ
# ==============================================================================
# Test verisi eşik seçiminde kullanılmamıştır.
# Validation kümesinde belirlenen eşik burada uygulanır.

print("\n--- Bağımsız Test Kümesi Değerlendiriliyor ---")

# Test kümesindeki spam olasılıkları
y_test_prob = classifier.predict(
    X_test,
    verbose=0
).flatten()

# Validation üzerinde seçilen eşik uygulanır
y_test_pred = (
    y_test_prob >= best_threshold
).astype(int)

print("Test tahminleri oluşturuldu.")

In [ ]:
# ==============================================================================
# 5.3. TEST PERFORMANS METRİKLERİ
# ==============================================================================
# Hesaplanan metrikler:
# - Doğruluk
# - Kesinlik
# - Duyarlılık
# - F1-Skoru
# - ROC-AUC
# - PR-AUC

test_accuracy = accuracy_score(
    y_test,
    y_test_pred
)

test_precision = precision_score(
    y_test,
    y_test_pred,
    zero_division=0
)

test_recall = recall_score(
    y_test,
    y_test_pred,
    zero_division=0
)

test_f1 = f1_score(
    y_test,
    y_test_pred,
    zero_division=0
)

test_roc_auc = roc_auc_score(
    y_test,
    y_test_prob
)

test_pr_auc = average_precision_score(
    y_test,
    y_test_prob
)

print("\n--- Nihai Test Sonuçları ---")
print(f"Karar eşiği : {best_threshold:.2f}")
print(f"Doğruluk    : {test_accuracy:.4f}")
print(f"Kesinlik    : {test_precision:.4f}")
print(f"Duyarlılık  : {test_recall:.4f}")
print(f"F1-Skoru    : {test_f1:.4f}")
print(f"ROC-AUC     : {test_roc_auc:.4f}")
print(f"PR-AUC      : {test_pr_auc:.4f}")

In [ ]:
# ==============================================================================
# 5.4. SINIFLANDIRMA RAPORU
# ==============================================================================
# Her sınıf için precision, recall ve F1-Skoru gösterilir.

print("\n--- Sınıflandırma Raporu ---")

print(
    classification_report(
        y_test,
        y_test_pred,
        target_names=[
            "Ham (0)",
            "Spam (1)"
        ],
        digits=4,
        zero_division=0
    )
)

In [ ]:
# ==============================================================================
# 5.5. KARMAŞIKLIK MATRİSİ
# ==============================================================================
# Matris sırası:
# TN | FP
# FN | TP

confusion_mat = confusion_matrix(
    y_test,
    y_test_pred
)

tn, fp, fn, tp = confusion_mat.ravel()

print("\n--- Karmaşıklık Matrisi Değerleri ---")
print(f"Doğru Negatif  (TN): {tn}")
print(f"Yanlış Pozitif (FP): {fp}")
print(f"Yanlış Negatif (FN): {fn}")
print(f"Doğru Pozitif  (TP): {tp}")

plt.figure(figsize=(6, 5))

sns.heatmap(
    confusion_mat,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=[
        "Ham",
        "Spam"
    ],
    yticklabels=[
        "Ham",
        "Spam"
    ]
)

plt.xlabel("Tahmin Edilen Sınıf")
plt.ylabel("Gerçek Sınıf")
plt.title("Karmaşıklık Matrisi")
plt.tight_layout()
plt.show()

In [ ]:
# ==============================================================================
# 5.6. FPR VE TPR HESAPLAMA
# ==============================================================================
# FPR:
# Ham e-postaların yanlışlıkla spam olarak işaretlenme oranı
#
# TPR:
# Spam e-postaların doğru tespit edilme oranı

false_positive_rate = (
    fp / (fp + tn)
    if (fp + tn) > 0
    else 0.0
)

true_positive_rate = (
    tp / (tp + fn)
    if (tp + fn) > 0
    else 0.0
)

false_negative_rate = (
    fn / (fn + tp)
    if (fn + tp) > 0
    else 0.0
)

print("\n--- Operasyonel Hata Oranları ---")
print(f"FPR: {false_positive_rate:.6f}")
print(f"TPR: {true_positive_rate:.6f}")
print(f"FNR: {false_negative_rate:.6f}")

In [ ]:
# ==============================================================================
# 5.7. EĞİTİM VE VALIDATION GRAFİKLERİ
# ==============================================================================
# Amaç:
# - Modelin öğrenme davranışını incelemek
# - Overfitting olup olmadığını gözlemlemek

plt.figure(figsize=(8, 5))

plt.plot(
    history.history["accuracy"],
    label="Eğitim Doğruluğu"
)

plt.plot(
    history.history["val_accuracy"],
    label="Validation Doğruluğu"
)

plt.xlabel("Epoch")
plt.ylabel("Doğruluk")
plt.title("Eğitim ve Validation Doğruluğu")
plt.legend()
plt.tight_layout()
plt.show()


plt.figure(figsize=(8, 5))

plt.plot(
    history.history["loss"],
    label="Eğitim Kaybı"
)

plt.plot(
    history.history["val_loss"],
    label="Validation Kaybı"
)

plt.xlabel("Epoch")
plt.ylabel("Kayıp")
plt.title("Eğitim ve Validation Kaybı")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ==============================================================================
# 5.8. TEST SONUÇLARINI KAYDETME
# ==============================================================================
test_results = pd.DataFrame([{
    "threshold": best_threshold,
    "accuracy": test_accuracy,
    "precision": test_precision,
    "recall": test_recall,
    "f1_score": test_f1,
    "roc_auc": test_roc_auc,
    "pr_auc": test_pr_auc,
    "tn": tn,
    "fp": fp,
    "fn": fn,
    "tp": tp,
    "fpr": false_positive_rate,
    "tpr": true_positive_rate,
    "fnr": false_negative_rate
}])

results_path = os.path.join(
    MODEL_SAVE_DIR,
    "test_results.csv"
)

test_results.to_csv(
    results_path,
    index=False
)

print(
    f"\nTest sonuçları kaydedildi: "
    f"{results_path}"
)